# Multi-energy multimode phase retrieval
Minimal use of `phase_retrieval_core_multienergy_multimode.py`.

In [4]:
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sys, os
from os.path import join
sys.path.append("/home/user/Moritz/msc/RB2004/code/FTH-Phase_Retrieval/library")
import phase_retrieval_core_multienergy as phr_me
import phase_retrieval_core_multienergy_multimode as pr

In [ ]:
holograms = np.load("data/energy_holograms.npy")  # (n_energy, nx, ny)
mask_pixel = np.load("data/mask_pixel.npy")
supportmask = np.load("data/supportmask.npy")

In [6]:
data = np.load(Path("/home/user/Moritz/msc/RB2004/data.npz"))
holograms = data["holograms"]
energy_ev = data["energy"]
mask_pixel = data["mask_pixel"]
supportmask = np.load("/home/user/Moritz/msc/RB2004/suppmask.npz.npy")

optical_constants = np.load(Path("/home/user/Moritz/msc/RB2004/data/refractive_index_constraints.npz"))
constraint_energy_ev = optical_constants["energy_ev"]
beta = optical_constants["beta"]
delta = optical_constants["delta"]

#if not np.allclose(energy_ev, constraint_energy_ev):
beta = np.interp(energy_ev, constraint_energy_ev, beta)
delta = np.interp(energy_ev, constraint_energy_ev, delta)

assert holograms.shape[0] == energy_ev.size == beta.size == delta.size
assert supportmask.shape == holograms.shape[1:]

In [8]:
import cupy as cp
import numpy as np

if not hasattr(cp, 'errstate'):
    cp.errstate = np.errstate

In [9]:
recipe = {
    # Per-energy multimode update schedule.
    "inner_mode": ["HAPRE", "ER"],       # Algorithm stages at every energy.
    "inner_Nit": [700, 50],               # Iterations in each stage.
    "outer_iterations": 100,              # Number of update-plus-projection cycles.
    "warmup_mode": ["HAPRE", "ER"],      # Independent pre-coupling stages.
    "warmup_Nit": 0,                      # Zero disables warmup.
    "shuffle_energies": True,             # Randomize energy order each outer cycle.
    "random_seed": None,                  # Seed for energy-order randomization.
    "beta_zero": 0.5,                     # Beta value(s), broadcast or one per stage.
    "beta_mode": "arctan",               # Beta schedule name(s) or arrays.
    "alpha_zero": 0.0,                    # TV strength; zero disables TV.
    "alpha_mode": "const",               # Alpha schedule name(s) or arrays.
    "TV_freq": 1e9,                       # TV update interval.
    "warmup_beta_zero": None,             # None inherits the inner setting.
    "warmup_beta_mode": None,             # None inherits the inner setting.
    "warmup_alpha_zero": None,            # None inherits the inner setting.
    "warmup_alpha_mode": None,            # None inherits the inner setting.
    "warmup_TV_freq": None,               # None inherits the inner setting.
    "plot_every": 1e9,                    # Error sampling/plot interval.
    "average_img": 1,                     # Number of best late iterates to average.
    "Fourier_last": True,                 # Finish each stage with its Fourier constraint.
    "final_fourier_constraint": True,     # Enforce measured summed modal intensity at the end.
    "hologram_intensity_cutoff_vmin": -1, # Percentile baseline subtraction.
    # Cross-energy projection applied independently to every mode.
    "projection_model": "rank1_spectral",           # "none", "svd", or "rank1_spectral".
    "rank": 1,                            # SVD residual rank.
    "projection_every": 1,                # Projection interval in outer cycles.
    "projection_relaxation": 1.0,         # Projection blending fraction.
    "projection_constraints_inside_support_only": False, # If True, apply joint projections only inside supportmask.
    "projection_start": 0,                # First cycle eligible for projection.
    "projection_static_mode": "mean",    # "mean", "first", or "none".
    "energy_weights": None,               # Positive weight per energy.
    "log_floor": 1e-12,                   # Magnitude floor before complex log.
    "spectral_constraint": "known_beta",       # Rank-one spectrum: free, KK, or known-beta modes.
    "energy_values": None,                # Strictly increasing energies for KK constraints.
    "known_beta_spectrum": beta,          # Known absorption-like spectrum.
    "known_delta_spectrum": None,         # Optional known dispersion-like spectrum.
    "absorption_part": "real",           # Absorption location in the complex coefficient.
    "kk_sign": 1.0,                       # KK sign convention.
    "kk_subtract_baseline": True,         # Remove endpoint baseline before KK.
    "kk_normalize_input": False,          # Normalize absorption before KK.
    "known_beta_normalization": "none",  # none, maxabs, l2, or std.
    "fit_known_beta_scale": True,         # Fit known-spectrum scale.
    "fit_known_beta_offset": True,        # Fit known-spectrum offset.
    # Multimode-only controls.
    "Nmodes": 3,                          # Number of mutually incoherent modes per energy.
    "mode_initialization_seed": 0,        # Seed used to break initial modal degeneracy.
}

fields, components, bsmasks, errors = (
    pr.multi_energy_phase_retrieval_algorithm(
        holograms,
        mask_pixel,
        supportmask,
        multi_energy_recipe=recipe,
    )
)

KeyboardInterrupt: 